In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import pickle
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
def rfefeature(indep_x,dep_y,n):
    rfelist=[]
    log_model=LogisticRegression(solver='lbfgs')
    svc=SVC(kernel = 'linear', random_state = 0)
    DT=DecisionTreeClassifier(criterion = 'gini', max_features='sqrt',splitter='best',random_state = 0)
    RF=RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
    rfemodellist=[log_model,svc,DT,RF]
    for i in rfemodellist:
        print(i)
        log_rfe=RFE(estimator=i, n_features_to_select=n)
        log_fit=log_rfe.fit(indep_x,dep_y)
        log_rfe_feature=log_fit.transform(indep_x)
        rfelist.append(log_rfe_feature)
    return rfelist
    

In [3]:
def split_scaler(indep_x,dep_y):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    x_train,x_test,y_train,y_test=train_test_split(indep_x,dep_y,test_size=0.25,random_state=42)
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    return x_train,x_test,y_train,y_test

In [4]:
def cm_prediction(classifier,x_test):
    y_pred=classifier.predict(x_test)
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(y_test,y_pred)
    from sklearn.metrics import classification_report
    report=classification_report(y_test,y_pred)
    from sklearn.metrics import accuracy_score
    Accuracy=accuracy_score(y_test,y_pred)
    return classifier,Accuracy,report,cm,x_test,y_test

In [31]:
def logistic(x_train,y_train,x_test):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GridSearchCV
    param_grid={'solver':['lbfgs','newton-cg','liblinear']}
    classifier=GridSearchCV(LogisticRegression(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [32]:
def svc(x_train,y_train,x_test):
    from sklearn.svm import SVC
    from sklearn.model_selection import GridSearchCV
    param_grid={'kernel':['rbf','linear','poly'],'gamma':['auto','scale']}
    classifier=GridSearchCV(SVC(probability=True),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [33]:
def Decision(x_train,y_train,x_test):
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.model_selection import GridSearchCV
    param_grid={'criterion':['gini','entropy','log_loss'],'splitter':['best','random']}
    classifier=GridSearchCV(DecisionTreeClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [34]:
def random(x_train,y_train,x_test):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import GridSearchCV
    param_grid={'criterion':['gini','entropy','log_loss'],'max_features':['sqrt','log2'],'n_estimators':[100]}
    classifier=GridSearchCV(RandomForestClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [35]:
def rfeclassification(acclog,accsvc,accdt,accrf):
    rfe_dataframe=pd.DataFrame(index=['Logistic','SVC','DT','RF'],columns=['Logistic','SVC','DecisionTree','RandomForest'])
    for number,idex in enumerate(rfe_dataframe.index):
        rfe_dataframe['Logistic'][idex]=acclog[number]
        rfe_dataframe['SVC'][idex]=accsvc[number]
        rfe_dataframe['DecisionTree'][idex]=accdt[number]
        rfe_dataframe['RandomForest'][idex]=accrf[number]
    return rfe_dataframe

In [36]:
dataset=pd.read_csv("preprocessed ILPD.csv")
df=dataset
df=pd.get_dummies(df,dtype=int,drop_first=True)
df

,Age,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G Ratio,Selector,Gender_Male
0,65,0.7,0.10,187.00,16.0,18,6.8,3.3,0.90,1,0
1,62,5.3,2.95,481.75,64.0,100,7.5,3.2,0.74,1,1
2,62,5.3,2.95,481.75,60.0,68,7.0,3.3,0.89,1,1
3,58,1.0,0.40,182.00,14.0,20,6.8,3.4,1.00,1,1
4,72,3.9,2.00,195.00,27.0,59,7.3,2.4,0.40,1,1
...,...,...,...,...,...,...,...,...,...,...,...
578,60,0.5,0.10,481.75,20.0,34,5.9,1.6,0.37,0,1
579,40,0.6,0.10,98.00,35.0,31,6.0,3.2,1.10,1,1
580,52,0.8,0.20,245.00,48.0,49,6.4,3.2,1.00,1,1
581,31,1.3,0.50,184.00,29.0,32,6.8,3.4,1.00,1,1


In [37]:
indep_x=df.drop('Selector',axis=1)
dep_y=df['Selector']

In [38]:
indep_x.shape
dep_y.shape
print(indep_x.shape)
print(type(indep_x))


(583, 10)
<class 'pandas.core.frame.DataFrame'>


In [39]:
rfelist=rfefeature(indep_x,dep_y,5)
acclog=[]
accsvc=[]
accdt=[]
accrf=[]
for i in rfelist:   
    x_train, x_test, y_train, y_test=split_scaler(i,dep_y)   
    classifier,Accuracy,report,x_test,y_test,cm=logistic(x_train,y_train,x_test)
    acclog.append(Accuracy)
    classifier,Accuracy,report,x_test,y_test,cm=svc(x_train,y_train,x_test)
    accsvc.append(Accuracy)
    classifier,Accuracy,report,x_test,y_test,cm=Decision(x_train,y_train,x_test)
    accdt.append(Accuracy)
    classifier,Accuracy,report,x_test,y_test,cm=random(x_train,y_train,x_test)
    accrf.append(Accuracy)
result=rfeclassification(acclog,accsvc,accdt,accrf)

LogisticRegression()


C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _che

SVC(kernel='linear', random_state=0)
DecisionTreeClassifier(max_features='sqrt', random_state=0)
RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Fitting 5 folds for each of 6 candidates, totalling 30 fits


ValueError: Expected a 2-dimensional container but got <class 'pandas.core.series.Series'> instead. Pass a DataFrame containing a single row (i.e. single sample) or a single column (i.e. single feature) instead.